# CS431 · Pilot sàng lọc Flip-TTA trên P-map (B1, 1 checkpoint) trên Kaggle T4

**Kiểm tra giả thuyết:** trung bình bản đồ xác suất của ảnh gốc và ảnh lật ngang (đã lật ngược về đúng tọa độ), trước hậu xử lý DB, có giảm FP / tăng Hmean của B1 `.40/.60` không?

**Phạm vi:** một checkpoint B1, **không train**, hậu xử lý giữ nguyên (`mask_040`, score 0,60), **không quét ngưỡng**. Mẫu cố định gồm 300 nguồn của split A, cả Clean và LR2 (600 ảnh). Spec khóa trước: `docs/research/janek/fliptta_spec_20261123_janek.md`.
Đây là pilot development, **không phải xác nhận độc lập**; TTA chưa mặc định được tính là đóng góp phương pháp mới.

**Hai cánh, cùng một lượt chạy, cùng GPU:**
- **F:** một forward trên ảnh gốc (baseline B1 `.40/.60`).
- **T:** P-map = trung bình của P0 và P1b (P-map của ảnh lật, lật ngược). Chỉ lật vùng nội dung, vùng pad giữ nguyên, không lấy mẫu lại.

**Kiểm căn chỉnh (cứng) chạy cho từng ảnh:** pad hằng, lật hai lần = nguyên bản, cùng metainfo, P-map hữu hạn cùng shape, và trung vị IoU đã căn phải lớn hơn đối chứng chưa lật ngược. Trượt thì **không diễn giải chất lượng**.

### Trước khi chạy
1. Kaggle → **Create → New Dataset**, tải `cs431-fliptta-pilot.zip` lên, để **Private**.
2. **File → Import Notebook**, chọn file này.
3. Cột phải: **Accelerator** `GPU T4 x2` (hoặc `GPU T4`), **Internet On**, **Add Input** → dataset vừa tạo.
4. Chạy **Ô 1 → Ô 4**, đọc kết quả chạy thử ở Ô 4, rồi mới chạy **Ô 5 → Ô 7**.

**Thời gian: chưa đo.** Ô 2: 5–40 phút. Ô 5: mỗi ảnh chạy F + T (khoảng 3 lượt forward) + kiểm căn chỉnh; Ô 4 in ước lượng **đo thật**.

**Dừng nếu:** Ô 4 báo kiểm căn chỉnh cứng sai, hoặc thời gian dự kiến vượt 3 giờ (báo lại Claude, không tự đổi cấu hình hay giảm số ảnh).

**Trạng thái:** notebook đã tạo và kiểm cú pháp ở máy local; **chưa chạy trên Kaggle**.

## Ô 1 · Cấu hình, tìm dataset, kiểm GPU

Kiểm SHA của mọi file trong dataset. Ghim **GPU 0** (`CUDA_VISIBLE_DEVICES=0`), vì code của Tuấn yêu cầu đúng một GPU.

In [ ]:
# ===================== Ô 1 · CẤU HÌNH =====================
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

PIN = dict(
    MONKEY_V2_URL='https://github.com/ncat218/MONKEY_V2.git',
    MONKEY_V2_COMMIT='dc19b85c8f73d56dd0510165069597363fe25380',
    MONKEYOCRV2_URL='https://github.com/Yuliang-Liu/MonkeyOCRv2.git',
    MONKEYOCRV2_COMMIT='da96afede92f198c19aca74e1e5f5aa7f4c26af2',
    TORCH='2.9.0', TORCHVISION='0.24.0', TORCHAUDIO='2.9.0', TORCH_INDEX='https://download.pytorch.org/whl/cu128',
    MMENGINE='0.10.7', MMCV='2.0.1', MMOCR='1.0.1', PILLOW='11.3.0', OPENCV='4.10.0.84',
    DB_PP_SHA256='9bfff8aaaca103dd08d7565f2b7a5410ab20a6427fe1f0baad5a5d04d207c8b8',
    BUNDLE_MANIFEST_SHA256='e06ef54e720b43bcb658fd734291fb1b6e4a08db13a8bd4048f16cfd3a2dc573',
)
W = Path('/kaggle/working'); OUT = W/'cs431_out'; LOGS = OUT/'logs'
T = Path('/tmp/cs431')
for p in (OUT, LOGS, T):
    p.mkdir(parents=True, exist_ok=True)

def fail(msg):
    raise SystemExit('LỖI: ' + msg)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

found = [p for p in Path('/kaggle/input').glob('**/KAGGLE_MANIFEST.json') if json.loads(p.read_text()).get('kind') == 'fliptta_pilot']
if not found:
    zips = [z for z in Path('/kaggle/input').glob('**/*.zip') if 'fliptta' in z.name.lower()]
    if not zips:
        fail('Không thấy dataset cs431-fliptta-pilot. Bấm "Add Input" và chọn dataset đó.')
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(T/'dataset')
    found = [p for p in (T/'dataset').glob('**/KAGGLE_MANIFEST.json') if json.loads(p.read_text()).get('kind') == 'fliptta_pilot']
if len(found) != 1:
    fail(f'Thấy {len(found)} dataset fliptta; chỉ gắn một.')
DS = found[0].parent
manifest = json.loads(found[0].read_text())
bad = [rel for rel, info in manifest['files'].items()
       if rel != 'KAGGLE_MANIFEST.json' and (not (DS/rel).is_file() or sha256(DS/rel) != info['sha256'])]
if bad:
    fail(f'{len(bad)} file thiếu hoặc sai SHA, ví dụ {bad[:3]}.')
print(f'Dataset: {DS} ({len(manifest["files"])} file, SHA khớp) · mẫu {manifest["n_sources"]} nguồn A')

CODE = T/'code'
if CODE.exists():
    shutil.rmtree(CODE)
shutil.copytree(DS/'code', CODE)
code_sha = hashlib.sha256((DS/'code/CODE_MANIFEST.json').read_bytes()).hexdigest()
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
import importlib
import mixab.colab_helpers_janek as ch
importlib.reload(ch)

gpus = (ch.output_of(['nvidia-smi', '--query-gpu=index,name,driver_version,memory.total', '--format=csv,noheader']) or '').splitlines()
if not gpus:
    fail('Không thấy GPU. Settings -> Accelerator -> GPU T4 x2 (hoặc GPU T4).')
print('GPU của máy:'); [print('  ', g) for g in gpus]
GPU = ch.gpu_label(gpus[0].split(',')[1].strip())
CONFIG_NOTE = f'{len(gpus)} GPU trên máy; chỉ dùng GPU 0 qua CUDA_VISIBLE_DEVICES=0'
print(CONFIG_NOTE)

PY = sys.executable
MV2, MOCR = T/'MONKEY_V2', T/'MonkeyOCRv2'
BUNDLE = MV2/'docs/handoff/artifacts/pr_quad_audit_v1'
WEIGHTS, CLEAN, DUMP, A2R = DS/'weights', DS/'clean', DS/'dump', DS/'a2r/B1_source_counts.npz'
ENCODER = T/'encoder'
if not (ENCODER/'model.safetensors').is_file():
    shutil.copytree(DS/'encoder', ENCODER, dirs_exist_ok=True)
PREPARED, CONFIG_PATH = T/'prepared', T/'inference_config_kaggle.py'
ENV = dict(MONKEY_V2_ROOT=str(MV2), PYTHONPATH=str(CODE), CUDA_VISIBLE_DEVICES='0')
print(f'Python {sys.version.split()[0]} · code manifest {code_sha[:12]}… · kết quả: {OUT}')

## Ô 2 · Cài môi trường (cùng phiên bản với lần chạy dump T4 trên Colab)

Thứ tự: torch 2.9.0 cu128 → mmengine 0.10.7 → **MMCV 2.0.1** (dùng wheel trong dataset nếu cùng phiên bản Python, nếu không thì **build từ source**) → `install.sh` của MonkeyOCRv2 (MMOCR 1.0.1 + add-on) → ghim Pillow 11.3.0 và OpenCV 4.10.0.84.
- Build MMCV có thể mất **20–40 phút** (ước lượng, chưa đo trên Kaggle). Wheel build xong được chép vào `/kaggle/working/cs431_out/wheels/`: **tải về và thêm vào dataset** để lần sau khỏi build.
- Cần `nvcc` CUDA **12.x** để build. Ô này dùng toolkit có sẵn trên Kaggle (ưu tiên 12.8); nếu không có thì cài `cuda-toolkit-12-8` (không cài driver).
- Kiểm cuối chạy trong tiến trình mới: phiên bản, `MonkeyOCRv2ViTAEBackbone` đã đăng ký, SHA của DBPostprocessor phải đúng bản Tuấn ghim.
- Nếu Kaggle báo cần *Restart*, restart rồi chạy lại **Ô 1** và **Ô 2** (phần đã cài sẽ bỏ qua).

In [ ]:
# ===================== Ô 2 · CÀI MÔI TRƯỜNG =====================
import platform, re
LOG = LOGS/f'install_{GPU}.log'
pyver = sys.version_info
if not ((3, 10) <= pyver[:2] <= (3, 13)):
    print(f'CẢNH BÁO: Python {platform.python_version()} chưa được kiểm với MMCV 2.0.1.')

def ver(name):
    return ch.package_version(name)

def pip(*args):
    ch.run([PY, '-m', 'pip', 'install', '-q', *args], LOG, title='pip install ' + ' '.join(args))

if ver('torch') != f"{PIN['TORCH']}+cu128" or ver('torchvision') != f"{PIN['TORCHVISION']}+cu128":
    pip(f"torch=={PIN['TORCH']}", f"torchvision=={PIN['TORCHVISION']}", '--index-url', PIN['TORCH_INDEX'])
if ver('torchaudio') != f"{PIN['TORCHAUDIO']}+cu128":
    pip('--no-deps', f"torchaudio=={PIN['TORCHAUDIO']}+cu128", '--index-url', PIN['TORCH_INDEX'])
cuda_ok = ch.output_of([PY, '-c', 'import torch;print(torch.version.cuda, torch.cuda.is_available())'], env=ENV)
print('torch CUDA:', cuda_ok)
if not cuda_ok or 'True' not in cuda_ok:
    fail('torch không dùng được CUDA (driver Kaggle quá cũ cho cu128?). Gửi log: ' + str(LOG))
if ver('mmengine') != PIN['MMENGINE']:
    pip(f"mmengine=={PIN['MMENGINE']}")
pip('setuptools<81', 'wheel', 'ninja')

def mmcv_ok():
    if ver('mmcv') != PIN['MMCV']:
        return False
    probe = subprocess.run([PY, '-c', 'from mmcv.ops import nms; print("IMPORT_NMS_OK")'], capture_output=True, text=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nMMCV import probe:\n' + probe.stdout + probe.stderr)
    return probe.returncode == 0 and 'IMPORT_NMS_OK' in probe.stdout.splitlines()

def find_toolkit():
    """CUDA 12.x toolkit with nvcc + cuda.h; prefer 12.8. torch only refuses a different MAJOR version."""
    import glob
    homes = ['/usr/local/cuda-12.8', os.environ.get('CUDA_HOME'), '/usr/local/cuda', *sorted(glob.glob('/usr/local/cuda-12*'))]
    nvcc = shutil.which('nvcc')
    if nvcc:
        homes.append(str(Path(nvcc).resolve().parents[1]))
    best = None
    for home in homes:
        if not home or not (Path(home)/'bin/nvcc').is_file() or not (Path(home)/'include/cuda.h').is_file():
            continue
        m = re.search(r'release (\d+)\.(\d+)', ch.output_of([str(Path(home)/'bin/nvcc'), '--version']) or '')
        if m and m.group(1) == '12':
            cand = (int(m.group(2)) == 8, int(m.group(2)), str(Path(home).resolve()))
            best = max(best, cand) if best else cand
    return best

tag = f'cp{pyver[0]}{pyver[1]}'
if not mmcv_ok():
    wheels = sorted((DS/'wheels').glob(f"mmcv-{PIN['MMCV']}-{tag}-*.whl")) + sorted(OUT.glob(f"wheels/mmcv-{PIN['MMCV']}-{tag}-*.whl"))
    if wheels:
        print('Cài MMCV từ wheel có sẵn:', wheels[0].name)
        pip('--no-deps', str(wheels[0]))
    else:
        tk = find_toolkit()
        if not tk:
            print('Không có CUDA toolkit 12.x. Cài cuda-toolkit-12-8 từ NVIDIA (không cài driver)...')
            distro = platform.freedesktop_os_release()
            repo = 'ubuntu' + distro.get('VERSION_ID', '22.04').replace('.', '')
            from urllib.request import urlretrieve
            keyring = T/'cuda-keyring_1.1-1_all.deb'
            urlretrieve(f'https://developer.download.nvidia.com/compute/cuda/repos/{repo}/x86_64/cuda-keyring_1.1-1_all.deb', keyring)
            ch.run(['dpkg', '-i', keyring], LOG)
            ch.run(['apt-get', 'update'], LOG)
            ch.run(['apt-get', 'install', '-y', '--no-install-recommends', 'cuda-toolkit-12-8'], LOG, env={'DEBIAN_FRONTEND': 'noninteractive'})
            tk = find_toolkit()
            if not tk:
                fail('Đã cài nhưng vẫn không thấy nvcc 12.x; xem ' + str(LOG))
        cuda_home = tk[2]
        print(f'Dùng CUDA toolkit {cuda_home} (12.{tk[1]})' + ('' if tk[0] else ' — khác 12.8, torch chỉ cảnh báo khác minor version'))
        src = T/'mmcv_src'
        if not (src/'setup.py').is_file():
            ch.run(['git', 'clone', '-b', f"v{PIN['MMCV']}", '--depth', '1', 'https://github.com/open-mmlab/mmcv.git', src], LOG)
        setup = (src/'setup.py').read_text()
        setup = setup.replace('-std=c++14', '-std=c++17')
        old_v = "        exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']"
        new_v = "        namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)\n    return namespace['__version__']"
        if old_v in setup:
            setup = setup.replace(old_v, new_v)
        elif new_v not in setup:
            fail('MMCV setup.py khác cấu trúc đã kiểm.')
        (src/'setup.py').write_text(setup)
        print(f'Build MMCV ({os.cpu_count()} CPU). Có thể mất lâu...')
        build_env = dict(CUDA_HOME=cuda_home, MMCV_WITH_OPS='1', MAX_JOBS=str(os.cpu_count() or 2),
                         PATH=f"{cuda_home}/bin:{os.environ.get('PATH', '')}", TORCH_CUDA_ARCH_LIST='7.5;8.0')
        wheel_tmp = T/'wheelhouse'
        ch.run([PY, '-m', 'pip', 'wheel', '-v', '--no-build-isolation', '--no-deps', '-w', wheel_tmp, src], LOG,
               env=build_env, title='build MMCV wheel')
        built = sorted(wheel_tmp.glob('mmcv-*.whl'))
        if not built:
            fail('Build MMCV không tạo ra wheel; xem ' + str(LOG))
        (OUT/'wheels').mkdir(exist_ok=True)
        shutil.copy2(built[-1], OUT/'wheels'/built[-1].name)
        print('Đã lưu wheel để tải về:', OUT/'wheels'/built[-1].name)
        pip('--no-deps', str(built[-1]))
    if not mmcv_ok():
        fail('MMCV chưa import được mmcv.ops.nms; xem ' + str(LOG))

if not (MOCR/'.git').is_dir():
    ch.run(['git', 'clone', PIN['MONKEYOCRV2_URL'], MOCR], LOG)
ch.run(['git', '-C', MOCR, 'checkout', '--detach', PIN['MONKEYOCRV2_COMMIT']], LOG)
addon = MOCR/'detection/mmocr'
mmocr_root = addon/'mmocr'
backbone_file = mmocr_root/'mmocr/models/common/backbones/monkeyocr_v2_vitae.py'
mmocr_loc = ch.output_of([PY, '-c', 'import mmocr,os;print(os.path.dirname(mmocr.__file__))'])
if not (ver('mmocr') == PIN['MMOCR'] and backbone_file.is_file() and mmocr_loc and str(mmocr_root) in mmocr_loc):
    if not (mmocr_root/'.git').is_dir():
        ch.run(['git', 'clone', '--branch', 'v1.0.1', '--depth', '1', 'https://github.com/open-mmlab/mmocr.git', mmocr_root], LOG)
    setup_path = mmocr_root/'setup.py'
    setup = setup_path.read_text()
    old_exec = "        exec(compile(f.read(), version_file, 'exec'))"
    new_exec = "        namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)"
    if old_exec in setup:
        setup = setup.replace(old_exec, new_exec)
        setup = setup.replace("locals()['short_version']", "namespace['short_version']").replace("locals()['__version__']", "namespace['__version__']")
        setup_path.write_text(setup)
    elif new_exec not in setup:
        fail('MMOCR setup.py khác cấu trúc đã kiểm.')
    ch.run(['bash', 'install.sh'], LOG, cwd=addon, title='MonkeyOCRv2 detection/mmocr/install.sh',
           env=dict(PATH=f"{Path(PY).parent}:{os.environ.get('PATH', '')}", PIP_VERBOSE='1'))
if ver('Pillow') != PIN['PILLOW']:
    pip(f"Pillow=={PIN['PILLOW']}")
if ver('opencv-python') != PIN['OPENCV'] or any(ver(n) for n in ('opencv-python-headless', 'opencv-contrib-python')):
    ch.run([PY, '-m', 'pip', 'uninstall', '-y', 'opencv-python-headless', 'opencv-contrib-python', 'opencv-contrib-python-headless'], LOG)
    pip('--force-reinstall', '--no-deps', f"opencv-python=={PIN['OPENCV']}")
if ver('shapely') is None:
    pip('shapely')

VERIFY = """
import hashlib, importlib.metadata as m, importlib.util, json, torch
from mmocr.utils import register_all_modules
from mmocr.registry import MODELS
from mmcv.ops import nms
register_all_modules(init_default_scope=True)
spec = importlib.util.find_spec('mmocr.models.textdet.postprocessors.db_postprocessor')
out = dict(versions={k: m.version(k) for k in ('torch','torchvision','mmengine','mmcv','mmdet','mmocr','Pillow','opencv-python','numpy','shapely')},
           cuda=torch.cuda.is_available(), device_count_visible=torch.cuda.device_count(), device=torch.cuda.get_device_name(0),
           backbone_registered=MODELS.get('MonkeyOCRv2ViTAEBackbone') is not None,
           db_postprocessor_sha256=hashlib.sha256(open(spec.origin,'rb').read()).hexdigest())
print(json.dumps(out))
"""
raw = ch.output_of([PY, '-c', VERIFY], env=ENV)
if not raw:
    fail('Kiểm cuối môi trường lỗi khi import; xem ' + str(LOG))
env_report = json.loads(raw.splitlines()[-1])
(OUT/f'env_{GPU}.json').write_text(json.dumps(dict(env_report, pins=PIN, gpus=gpus, config=CONFIG_NOTE), indent=1))
print(json.dumps(env_report, indent=1))
problems = []
if env_report['db_postprocessor_sha256'] != PIN['DB_PP_SHA256']:
    problems.append('DBPostprocessor khác bản Tuấn ghim')
if not env_report['backbone_registered']:
    problems.append('MonkeyOCRv2ViTAEBackbone chưa đăng ký')
if env_report['versions']['Pillow'] != PIN['PILLOW']:
    problems.append('Pillow không phải 11.3.0 (cần để tạo LR2 đúng byte)')
if env_report['device_count_visible'] != 1:
    problems.append('Tiến trình đo thấy nhiều hơn 1 GPU')
if problems:
    fail('Môi trường chưa đạt: ' + '; '.join(problems))
print('\nÔ 2 xong.')

## Ô 3 · Repo của Tuấn, config, checkpoint B1, ảnh LR2 của mẫu

Clone `MONKEY_V2` @ `dc19b85`, vá config (chỉ đường dẫn encoder), kiểm SHA B1, tạo 300 ảnh LR2 bằng `native_lr2` của Tuấn (kiểm SHA từng ảnh), chạy test CPU.

In [ ]:
# ===================== Ô 3 · DỮ LIỆU =====================
LOG = LOGS/f'data_{GPU}.log'
if not (MV2/'.git').is_dir():
    ch.run(['git', 'clone', PIN['MONKEY_V2_URL'], MV2], LOG, env=dict(GIT_LFS_SKIP_SMUDGE='1'))
ch.run(['git', '-C', MV2, 'checkout', '--detach', PIN['MONKEY_V2_COMMIT']], LOG, env=dict(GIT_LFS_SKIP_SMUDGE='1'))
if ch.output_of(['git', '-C', MV2, 'status', '--porcelain', '--untracked-files=no']):
    fail('Checkout MONKEY_V2 bị sửa.')
if sha256(BUNDLE/'bundle_manifest.json') != PIN['BUNDLE_MANIFEST_SHA256']:
    fail('bundle_manifest.json khác bản đã khóa.')
CONFIG_PATH.write_text(ch.patch_inference_config((MV2/'docs/handoff/inference_config.py').read_text(encoding='utf-8'), ENCODER), encoding='utf-8')
m = json.loads((WEIGHTS/'merge_manifest.json').read_text())
b1 = next(x for x in m['models'] if x['alpha'] == '1')
from pathlib import PureWindowsPath
if sha256(WEIGHTS/PureWindowsPath(b1['checkpoint']).name) != b1['checkpoint_sha256']:
    fail('alpha_1p00.pth (B1) sai SHA.')
ch.run([PY, '-m', 'mixab.fliptta_janek', 'prepare', '--bundle', BUNDLE, '--clean-root', CLEAN, '--prepared-root', PREPARED,
        '--output', OUT], LOG, cwd=CODE, env=ENV, title='fliptta prepare')
ch.run([PY, '-m', 'unittest', 'discover', '-s', 'tests', '-t', '.'], LOG, cwd=CODE, env=ENV, title='mixab unit tests')
print('\nÔ 3 xong.')

## Ô 4 · Chạy thử 3 nguồn (6 ảnh): kiểm căn chỉnh, bộ nhớ, thời gian

- Chạy F và T trên 3 nguồn, in **kiểm căn chỉnh cứng**. Nếu pad/lật/metainfo sai thì ô dừng ngay.
- In thời gian từng cánh (ms), IoU đã căn so với đối chứng, thời gian **dự kiến toàn pilot** (đo thật), bộ nhớ đỉnh.
- Dự kiến vượt 3 giờ thì **dừng và báo Claude**.

In [ ]:
# ===================== Ô 4 · CHẠY THỬ =====================
SMOKE = OUT/'fliptta_smoke'
if not (SMOKE/'infer_summary.json').is_file():
    ch.run([PY, '-m', 'mixab.fliptta_janek', 'infer', '--bundle', BUNDLE, '--clean-root', CLEAN, '--prepared-root', PREPARED,
            '--model-config', CONFIG_PATH, '--weights-dir', WEIGHTS, '--limit-sources', '3', '--shard', '6',
            '--output', SMOKE], LOGS/'fliptta_smoke.log', cwd=CODE, env=ENV, title='fliptta smoke')
s = json.loads((SMOKE/'infer_summary.json').read_text())
import gzip
recs = [json.loads(l) for p in sorted(SMOKE.glob('shard_*.jsonl.gz')) for l in gzip.open(p, 'rt')]
print(f"Kiểm căn chỉnh: {s['alignment_gate']}")
print(f"{s['seconds_per_task']:.1f} s/ảnh → dự kiến toàn pilot (600 ảnh): {s['projected_minutes_full_pilot']:.0f} phút")
for r in recs:
    c = r['check']
    print(f"  {r['condition']:5} {r['size']} → tensor {r['img_shape']} · F {r['F']['ms']:.0f} ms ({len(r['F']['raw'])} ứng viên), "
          f"T {r['T']['ms']:.0f} ms ({len(r['T']['raw'])}) · thứ tự {r['order']} · IoU căn {c['iou_aligned']} / đối chứng {c['iou_control']} "
          f"· pad hằng {c['pad_constant']} · lật 2 lần {c['flip_twice_identity']} · cùng metainfo {c['same_metainfo']} · đỉnh {r['peak_mib']:.0f} MiB")
if not s['alignment_gate']['passed']:
    print('\nDỪNG: kiểm căn chỉnh không đạt. Báo Claude trước khi chạy Ô 5.')
elif s['projected_minutes_full_pilot'] > 180:
    print('\nDỪNG: dự kiến vượt 3 giờ. Báo Claude trước khi chạy Ô 5.')
else:
    print('\nĐược chạy Ô 5.')

## Ô 5 · Chạy pilot đủ 600 ảnh (GPU)

- Lưu sau mỗi 20 ảnh vào `cs431_out/fliptta_infer/`. Bị ngắt **trong cùng phiên** thì chạy lại ô này, ảnh đã xong sẽ bỏ qua.
- Nếu phiên Kaggle kết thúc, `/kaggle/working` mất. Trước khi dừng, chạy Ô 7 để tải phần đã xong.
- Kiểm cứng căn chỉnh chạy cho từng ảnh; sai thì ô dừng và báo ảnh nào.

In [ ]:
# ===================== Ô 5 · PILOT =====================
INF = OUT/'fliptta_infer'
ch.run([PY, '-m', 'mixab.fliptta_janek', 'infer', '--bundle', BUNDLE, '--clean-root', CLEAN, '--prepared-root', PREPARED,
        '--model-config', CONFIG_PATH, '--weights-dir', WEIGHTS, '--output', INF],
       LOGS/f'fliptta_infer_{time.strftime("%Y%m%d-%H%M%S")}.log', cwd=CODE, env=ENV, title='fliptta pilot infer')
print(json.loads((INF/'infer_summary.json').read_text()))

## Ô 6 · Đánh giá (CPU, chưa đo)

Chấm F và T bằng `CandidateMatcher` của Tuấn. Báo Clean và LR2: Hmean, TP/FP/FN, tiny, recall theo bin; Δ so với F kèm CI theo nguồn; chênh lệch theo từng nguồn (`per_source.csv`); latency thực đo; nhất quán của F với dump T4 và A2r. Nhãn kết quả tính theo luật khóa (spec mục 6).

In [ ]:
# ===================== Ô 6 · ĐÁNH GIÁ =====================
EV = OUT/'fliptta_eval'
ch.run([PY, '-m', 'mixab.fliptta_janek', 'evaluate', '--bundle', BUNDLE, '--infer-dir', OUT/'fliptta_infer', '--dump', DUMP,
        '--a2r', A2R, '--zoom-sample', DS/'ref/zoom_sample.json', '--output', EV], LOGS/'fliptta_eval.log', cwd=CODE, env=ENV,
       title='fliptta evaluate')
r = json.loads((EV/'fliptta_result.json').read_text())
print(f"Nguồn: {r['sources']} (đủ: {r['complete']}) · căn chỉnh: {r['alignment_gate']}")
print(f"Nhất quán F: {r['consistency']} · chồng lấn mẫu zoom (mô tả): {r.get('overlap_with_zoom_sample')}")
for cond in ('lr2', 'clean'):
    c = r['conditions'][cond]
    for arm in ('F', 'T'):
        t = c[arm]
        print(f"  {cond:5} {arm}  H {t['hmean']*100:.2f}  TP {t['tp']} FP {t['fp']} FN {t['fn']}  recall {t['tp']/t['gt']*100:.2f}  tiny {t['tiny_recall']*100:.1f}")
    d = c['delta']
    print(f"  {cond:5} T − F: {d['delta_pp']:+.2f} pp, CI {d['ci95_pp'][0]:+.2f}..{d['ci95_pp'][1]:+.2f} · tiny {c['delta_tiny_pp']:+.2f} pp · "
          f"nguồn lên/xuống/bằng {c['per_source']['up']}/{c['per_source']['down']}/{c['per_source']['same']} · latency {c['latency_ms']}")
print('\nCờ:', r['flags'], '\nNhãn:', r['outcome'])

## Ô 7 · Đóng gói kết quả

Tạo `/kaggle/working/cs431_fliptta_<GPU>_<thời gian>.zip` gồm toàn bộ `cs431_out` (trừ wheel). Tải ở khung **Output** và gửi lại Claude (hoặc báo Claude tải bằng Kaggle API).

In [ ]:
# ===================== Ô 7 · ĐÓNG GÓI =====================
pack = W/f'cs431_fliptta_{GPU}_{time.strftime("%Y%m%d-%H%M%S")}.zip'
with zipfile.ZipFile(pack, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix != '.whl':
            zf.write(p, p.relative_to(OUT).as_posix())
    zf.writestr('code_manifest_sha256.txt', code_sha + '\n')
print('Đã đóng gói:', pack, f'({pack.stat().st_size/2**20:.1f} MiB)')